# On the very first run, the pipeline performs a FULL load; all subsequent runs switch to INCREMENTAL mode.

In [0]:
import requests
import time
import json
from datetime import datetime, timezone
from typing import Optional, List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import Row

# ================= SECRETS =================

password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"

# ================= CONFIG =================

CATALOG = "elixir"
BRONZE_SCHEMA = "brz"
META_SCHEMA = "meta"
BATCH_SIZE    = 10000

ENDPOINTS = [
    {
        "name": "incidents",
        "url": "https://elixir.msf.org/tas/api/incidents",
        "fields": None,
        "incremental_field": "modificationDate",
        "bronze_table": "elixir.brz.topdesk_incidents_raw",
        "pagination": "OFFSET"
    },

    {
        "name": "time_registrations",
        "url": "https://elixir.msf.org/tas/api/incidents/timeregistrations",
        "fields":"timeSpent,operator.id,creationDate,parent.id,notes,modificationDate",
        "incremental_field": "modificationDate",
        "bronze_table": "elixir.brz.topdesk_timeregistrations_raw",
        "pagination": "NEXT"
         
    },

    {
        "name": "priorities",
        "url": "https://elixir.msf.org/tas/api/incidents/priorities",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_priorities_raw",
        "pagination": "SINGLE"

    },

    {
        "name": "statuses",
        "url": "https://elixir.msf.org/tas/api/incidents/statuses",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_statuses_raw",
        "pagination": "SINGLE"

    },

    {
        "name": "urgencies",
        "url": "https://elixir.msf.org/tas/api/incidents/urgencies",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_urgenciess_raw",
        "pagination": "SINGLE"
    },
    
    {
        "name": "categories",
        "url": "https://elixir.msf.org/tas/api/incidents/categories",
        "fields":None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_categories_raw",
        "pagination": "SINGLE"
    },
     
    {
        "name": "subcategories",
        "url": "https://elixir.msf.org/tas/api/incidents/subcategories",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_subcategories_raw",
        "pagination": "SINGLE"
    },

    {
        "name": "closure_codes",
        "url": "https://elixir.msf.org/tas/api/incidents/closure_codes",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_incident_closurecodes_raw",
        "pagination": "SINGLE"
    },
    

    {
        "name": "branches",
        "url": "https://elixir.msf.org/tas/api/branches",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_branches_raw",
        "pagination": "SINGLE"
        
    },

    {
        "name": "budgetholders",
        "url": "https://elixir.msf.org/tas/api/budgetholders",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_budgetholders_raw",
        "pagination": "SINGLE"
    },

    {
        "name": "operators",
        "url": "https://elixir.msf.org/tas/api/operators",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_operators_raw",
        "pagination": "OFFSET"
    },
    
    {
        "name": "operatorgroup",
        "url": "https://elixir.msf.org/tas/api/operatorgroups",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_operatorgroup_raw",
        "pagination": "OFFSET"
    },

    {
        "name": "impacts",
        "url": "https://elixir.msf.org/tas/api/changes/impacts",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_changes_impacts_raw",
        "pagination": "SINGLE"
    },

    {
        "name": "operatorchanges",
        "url": "https://elixir.msf.org/tas/api/operatorChanges",
        "fields": "id,incident.id,changeType,status,category.id,category.name,subcategory.id,processingStatus,modificationDate",
        "incremental_field": "modificationDate",
        "bronze_table": "elixir.brz.topdesk_operator_changes_raw",
        "pagination": "NEXT"
    },

    {
        "name": "changeactivities",
        "url": "https://elixir.msf.org/tas/api/operatorChangeActivities",
        "fields": "timeSpent,id,category.id,lastModificationDate",
        "incremental_field": "lastModificationDate",
        "bronze_table": "elixir.brz.topdesk_operator_changeActivities_raw",
        "pagination": "NEXT"
    }
 ]




AUTH = (username, password)

PAGE_SIZE = 100
MAX_WORKERS = 10
MAX_RETRIES = 3
REQUEST_TIMEOUT = 30

LOAD_TYPE = "INCREMENTAL"   # "FULL" or "INCREMENTAL"
STATE_TABLE  = f"{CATALOG}.{META_SCHEMA}.pipeline_state"

PIPELINE_NAME = "topdesk_ingestion"


In [0]:
def format_ts(dt: datetime) -> str:
    """
    Format a datetime object to an ISO 8601 string with milliseconds and 'Z' (UTC) suffix.

    Args:
        dt (datetime): The datetime object to format.

    Returns:
        str: ISO 8601 formatted string (e.g., '2024-03-04T12:34:56.789Z').
    """
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def parse_ts(value: Optional[str]) -> Optional[datetime]:
    """
    Parse an ISO 8601 timestamp string to a datetime object.

    Args:
        value (Optional[str]): Timestamp string (may end with 'Z' for UTC).

    Returns:
        Optional[datetime]: Parsed datetime object, or None if input is None/empty.
    """
    if not value:
        return None

    # Convert 'Z' suffix to '+00:00' for Python compatibility
    if value.endswith("Z"):
        value = value.replace("Z", "+00:00")

    return datetime.fromisoformat(value)


def build_url(endpoint: dict, query: Optional[str]) -> str:
    """
    Construct a URL for the API endpoint, including optional fields and query parameters.

    Args:
        endpoint (dict): Endpoint configuration dictionary.
        query (Optional[str]): Query string for filtering results.

    Returns:
        str: Complete URL with query parameters.
    """
    base = endpoint["url"]
    params = []

    if endpoint.get("fields"):
        params.append(f"fields={endpoint['fields']}")

    if query:
        params.append(f"query={query}")

    if params:
        return base + "?" + "&".join(params)

    return base


# ============================================================
# STATE MANAGEMENT
# ============================================================

def get_last_run(endpoint_name: str) -> Optional[str]:
    """
    Retrieve the last successful run timestamp for a given endpoint from the state table.

    Args:
        endpoint_name (str): Name of the API endpoint.

    Returns:
        Optional[str]: Last run timestamp as a string, or None if not found or in FULL mode.
    """
    if LOAD_TYPE == "FULL":
        return None

    rows = spark.sql(f"""
        SELECT last_run
        FROM {STATE_TABLE}
        WHERE pipeline = '{PIPELINE_NAME}'
        AND endpoint = '{endpoint_name}'
        LIMIT 1
    """).collect()

    return rows[0]["last_run"] if rows else None


def ensure_state_table_exists():
    """
    Ensure the pipeline state table exists in the metastore.
    Creates the table if it does not exist.
    """
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            pipeline STRING,
            endpoint STRING,
            last_run STRING
        )
        USING DELTA
    """)


def update_last_run(endpoint_name: str, ts: str):
    """
    Update the last run timestamp for a given endpoint in the state table.

    Args:
        endpoint_name (str): Name of the API endpoint.
        ts (str): New last run timestamp (ISO 8601 string).
    """
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} t
        USING (
            SELECT '{PIPELINE_NAME}' AS pipeline,
                   '{endpoint_name}' AS endpoint,
                   '{ts}' AS last_run
        ) s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


# ============================================================
# QUERY
# ============================================================

def build_query(last_run: Optional[str], endpoint: dict) -> Optional[str]:
    """
    Build a query string for incremental data extraction based on the last run timestamp.

    Args:
        last_run (Optional[str]): Last run timestamp (ISO 8601 string).
        endpoint (dict): Endpoint configuration dictionary.

    Returns:
        Optional[str]: Query string for API filtering, or None for full load.
    """
    inc_field = endpoint.get("incremental_field")

    if LOAD_TYPE == "FULL" or not last_run or not inc_field:
        return None

    return f"{inc_field}>'{last_run}'"


# ============================================================
# OFFSET PAGINATION (PARALLEL)
# ============================================================

def fetch_page(session: requests.Session, endpoint: dict, start: int, query: Optional[str]) -> list:
    """
    Fetch a single page of results from an API endpoint using offset pagination.

    Args:
        session (requests.Session): Authenticated requests session.
        endpoint (dict): Endpoint configuration dictionary.
        start (int): Offset to start fetching records from.
        query (Optional[str]): Query string for filtering results.

    Returns:
        list: List of records (dicts) from the API response.
    """
    params = {
        "start": start,
        "page_size": PAGE_SIZE
    }

    if query:
        params["query"] = query

    if endpoint.get("fields"):
        params["fields"] = endpoint["fields"]

    resp = session.get(endpoint["url"], params=params, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    if not resp.text.strip():
        return []

    return resp.json()


def stream_endpoint_to_bronze(endpoint: dict, query: Optional[str]):
    """
    Stream data from an API endpoint to a bronze Delta table, handling pagination and batching.

    Args:
        endpoint (dict): Endpoint configuration dictionary.
        query (Optional[str]): Query string for filtering results.

    Returns:
        tuple: (total records ingested, max timestamp as ISO 8601 string or None)
    """
    endpoint_name = endpoint["name"]
    inc_field = endpoint.get("incremental_field")
    pagination_type = endpoint["pagination"]

    if pagination_type not in {"SINGLE", "OFFSET", "NEXT"}:
        raise Exception(f"Invalid pagination type: {pagination_type}")

    print(f"[INFO] {endpoint_name} pagination: {pagination_type}")

    total = 0
    batch = []
    max_ts = None

    # SINGLE: No pagination, fetch all at once
    if pagination_type == "SINGLE":
        url = build_url(endpoint, query)

        with requests.Session() as session:
            session.auth = AUTH
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()

            payload = resp.json()
            # Handle both list and dict with "results" key
            if isinstance(payload, dict):
                payload = payload.get("results", [])
            else:
                payload = payload

            for rec in payload:
                # Track the maximum timestamp for incremental loads
                if inc_field:
                    value = rec.get(inc_field)
                    if value:
                        parsed = parse_ts(value)
                        if parsed and (max_ts is None or parsed > max_ts):
                            max_ts = parsed
                batch.append(rec)
                total += 1

    # NEXT: Use "next" links for pagination
    elif pagination_type == "NEXT":
        url = build_url(endpoint, query)

        with requests.Session() as session:
            session.auth = AUTH

            while url:
                resp = session.get(url, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()

                payload = resp.json()
                results = payload.get("results", [])

                for rec in results:
                    if inc_field:
                        value = rec.get(inc_field)
                        if value:
                            parsed = parse_ts(value)
                            if parsed and (max_ts is None or parsed > max_ts):
                                max_ts = parsed

                    batch.append(rec)
                    total += 1

                if len(batch) >= BATCH_SIZE:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                print(f"[INFO] {endpoint_name} fetched={total}")

                url = payload.get("next")

    # OFFSET: Use offset and page_size for pagination
    else:  # OFFSET
        next_start = 0

        with requests.Session() as session:
            session.auth = AUTH

            while True:
                data = fetch_page(session, endpoint, next_start, query)

                if not data:
                    break

                for rec in data:
                    if inc_field:
                        value = rec.get(inc_field)
                        if value:
                            parsed = parse_ts(value)
                            if parsed and (max_ts is None or parsed > max_ts):
                                max_ts = parsed

                    batch.append(rec)
                    total += 1

                if len(batch) >= BATCH_SIZE:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                if len(data) < PAGE_SIZE:
                    break

                next_start += PAGE_SIZE

                print(f"[INFO] {endpoint_name} fetched={total}")

    if batch:
        write_to_bronze(batch, endpoint)

    return total, format_ts(max_ts) if max_ts else None


def write_to_bronze(incidents: list, endpoint: dict) -> None:
    """
    Write a batch of raw incident records to the bronze Delta table.

    Args:
        incidents (list): List of incident records (dicts) to write.
        endpoint (dict): Endpoint configuration dictionary (contains bronze_table name).
    """
    bronze_table = endpoint["bronze_table"]
    now = datetime.now(timezone.utc).isoformat()

    rows = [
        Row(
            raw_json=json.dumps(rec),
            ingestion_time=now
        )
        for rec in incidents
    ]

    df = spark.createDataFrame(rows)

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )

    print(f"[INFO] Written {len(incidents)} records to {bronze_table}")


def main():
    """
    Main entry point for the TOPdesk ingestion pipeline.

    Orchestrates the following steps for each endpoint:
        1. Ensure state table exists.
        2. Retrieve last run timestamp.
        3. Build query for incremental loads.
        4. Stream data from API to bronze table.
        5. Update last run timestamp in state table.

    Supports both FULL and INCREMENTAL loads, and handles different pagination types.
    """
    print(f"[INFO] Running pipeline in {LOAD_TYPE} mode")

    ensure_state_table_exists()

    for endpoint in ENDPOINTS:
        print("=" * 60)

        endpoint_name = endpoint["name"]
        print(f"[PIPELINE] {endpoint_name}")

        last_run = get_last_run(endpoint_name)

        query = build_query(last_run, endpoint)
        if query:
            print(f"[INFO] Filter: {query}")
        else:
            print("[INFO] FULL load (no filter)")

        total, max_ts = stream_endpoint_to_bronze(endpoint, query)

        print(f"[DONE] {endpoint_name} -> {total} records")

        # Update state only for endpoints with incremental fields
        if endpoint.get("incremental_field") and max_ts:
            update_last_run(endpoint_name, max_ts)
            print(f"[INFO] Updated last_run → {max_ts}")

In [0]:
def format_ts(dt: datetime) -> str:
    # Format datetime to ISO 8601 string with milliseconds and Zulu (UTC) suffix
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"



def parse_ts(value: Optional[str]) -> Optional[datetime]:
    if not value:
        return None

    # normalize Z → +00:00 so Python can parse it
    if value.endswith("Z"):
        value = value.replace("Z", "+00:00")

    return datetime.fromisoformat(value)
    
def build_url(endpoint, query):
    base = endpoint["url"]

    params = []

    if endpoint.get("fields"):
        params.append(f"fields={endpoint['fields']}")

    if query:
        params.append(f"query={query}")

    if params:
        return base + "?" + "&".join(params)

    return base

# ============================================================
# STATE MANAGEMENT
# ============================================================

def get_last_run(endpoint_name: str):
    if LOAD_TYPE == "FULL":
        return None

    rows = spark.sql(f"""
        SELECT last_run
        FROM {STATE_TABLE}
        WHERE pipeline = '{PIPELINE_NAME}'
        AND endpoint = '{endpoint_name}'
        LIMIT 1
    """).collect()

    return rows[0]["last_run"] if rows else None


def ensure_state_table_exists():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            pipeline STRING,
            endpoint STRING,
            last_run STRING
        )
        USING DELTA
    """)



def update_last_run(endpoint_name: str, ts: str):
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} t
        USING (
            SELECT '{PIPELINE_NAME}' AS pipeline,
                   '{endpoint_name}' AS endpoint,
                   '{ts}' AS last_run
        ) s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


# ============================================================
# QUERY
# ============================================================


def build_query(last_run, endpoint):
    inc_field = endpoint.get("incremental_field")

    if LOAD_TYPE == "FULL" or not last_run or not inc_field:
        return None

    #return f"{inc_field}>{last_run}"
    return f"{inc_field}>'{last_run}'"




# ============================================================
# OFFSET PAGINATION (PARALLEL)
# ============================================================


def fetch_page(session, endpoint, start, query):
    params = {
        "start": start,
        "page_size": PAGE_SIZE
    }

    if query:
        params["query"] = query

    if endpoint.get("fields"):
        params["fields"] = endpoint["fields"]

    resp = session.get(endpoint["url"], params=params, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    if not resp.text.strip():
       return []

    return resp.json()
    #return resp.json()

def stream_endpoint_to_bronze(endpoint, query):
    endpoint_name = endpoint["name"]
    inc_field = endpoint.get("incremental_field")
    pagination_type = endpoint["pagination"]
    
    if pagination_type not in {"SINGLE", "OFFSET", "NEXT"}:
        raise Exception(f"Invalid pagination type: {pagination_type}")

    print(f"[INFO] {endpoint_name} pagination: {pagination_type}")

    total = 0
    batch = []
    max_ts = None

    # =========================================================
    # SINGLE (NO PAGINATION)
    # =========================================================
    if pagination_type == "SINGLE":
        url = build_url(endpoint, query)

        with requests.Session() as session:
            session.auth = AUTH
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()

            payload = resp.json()
            # Handle both list and {"results:[.....]"}
            if isinstance(payload, dict):
                payload = payload.get("results", [])
            else:
                payload = payload

            for rec in payload:
                # track max timestamp safely
                # if LOAD_TYPE != "FULL" and inc_field:
                if inc_field:
                    value = rec.get(inc_field)

                    if value:
                        parsed = parse_ts(value)

                        if parsed and (max_ts is None or parsed > max_ts):
                            max_ts = parsed
                batch.append(rec)
                total += 1


    # =========================================================
    # NEXT PAGINATION
    # =========================================================
    elif pagination_type == "NEXT":
        url = build_url(endpoint, query)

        with requests.Session() as session:
            session.auth = AUTH

            while url:
                resp = session.get(url, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()

                payload = resp.json()
                results = payload.get("results", [])

                for rec in results:
                    if inc_field:
                        value = rec.get(inc_field)

                        if value:
                            parsed = parse_ts(value)

                            if parsed and (max_ts is None or parsed > max_ts):
                                max_ts = parsed
                        

                    batch.append(rec)
                    total += 1

                if len(batch) >= BATCH_SIZE:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                print(f"[INFO] {endpoint_name} fetched={total}")

                url = payload.get("next")

    # =========================================================
    # OFFSET PAGINATION
    # =========================================================
    else: # OFFSET
        next_start = 0

        with requests.Session() as session:
            session.auth = AUTH

            while True:
                data = fetch_page(session, endpoint, next_start, query)

                if not data:
                    break

                for rec in data:
                    if inc_field:
                        value = rec.get(inc_field)

                        if value:
                            parsed = parse_ts(value)

                            if parsed and (max_ts is None or parsed > max_ts):
                                max_ts = parsed

                    batch.append(rec)
                    total += 1

                if len(batch) >= BATCH_SIZE:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                if len(data) < PAGE_SIZE:
                    break

                next_start += PAGE_SIZE

                print(f"[INFO] {endpoint_name} fetched={total}")

    if batch:
        write_to_bronze(batch, endpoint)

    return total, format_ts(max_ts) if max_ts else None


def write_to_bronze(incidents: List[Dict], endpoint) -> None:
    """Write raw incidents into bronze Delta table."""
    bronze_table = endpoint["bronze_table"]
    now = datetime.now(timezone.utc).isoformat()

    rows = [
        Row(
            raw_json=json.dumps(rec),
            ingestion_time=now
        )
        for rec in incidents
    ]

    df = spark.createDataFrame(rows)

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )

    print(f"[INFO] Written {len(incidents)} records to {bronze_table}")


def main():
    """
    ============================================================
    TOPDESK INGESTION PIPELINE
    ============================================================

    This pipeline ingests multiple TOPdesk API endpoints into
    Bronze Delta tables.

    Features:
    - Supports FULL and INCREMENTAL loads
    - Handles multiple endpoints with different schemas
    - Supports both OFFSET and NEXT pagination styles
    - Writes data in batches for performance and stability
    - Tracks incremental state per endpoint

    Flow per endpoint:
        1. Read last_run from state table
        2. Build query filter (if incremental)
        3. Fetch data from API
        4. Write to Bronze in batches
        5. Update last_run checkpoint

    ============================================================
    """
    print(f"[INFO] Running pipeline in {LOAD_TYPE} mode")

    ensure_state_table_exists()

    for endpoint in ENDPOINTS:
        print("=" * 60)

        endpoint_name = endpoint["name"]
        print(f"[PIPELINE] {endpoint_name}")

        last_run = get_last_run(endpoint_name)

        query = build_query(last_run, endpoint)
        if query:
            print(f"[INFO] Filter: {query}")
        else:
            print("[INFO] FULL load (no filter)")

        total, max_ts = stream_endpoint_to_bronze(endpoint, query)

        print(f"[DONE] {endpoint_name} -> {total} records")

        # update state only for incremental endpoints
        if endpoint.get("incremental_field") and max_ts:
            update_last_run(endpoint_name, max_ts)
            print(f"[INFO] Updated last_run → {max_ts}")


In [0]:
if __name__ == "__main__":
    main()

## The codes below are just for testing the APIs before implementations

In [0]:

url = "https://elixir.msf.org/tas/api/operatorChanges"
#url = "https://elixir.msf.org/tas/api/incidents/closure_codes"

with requests.Session() as s:
    s.auth = AUTH
    r = s.get(url)
    print(r.status_code)
    #print(r.text)
    #print(r.json())

In [0]:
url = "https://elixir.msf.org/tas/api/incidents/closure_codes"
with requests.Session() as session:
    session.auth = AUTH

    r1 = session.get(
        url,
        params={"start": 0, "page_size": 1},
        timeout=REQUEST_TIMEOUT
    )
    r1.raise_for_status()
    data1 = r1.json()

    r2 = session.get(
        url,
        params={"start": 1, "page_size": 100},
        timeout=REQUEST_TIMEOUT
    )
    r2.raise_for_status()
    data2 = r2.json()

print("page_size=1 ->", len(data1))
print("page_size=100 ->", len(data2))
print("First IDs equal?:", data1[0]["id"] == data2[0]["id"])